# Lab 02: Training a Custom Model


**Objective of this lab**: training a small custom model on the Tiny-ImageNet dataset.

## Dataset preparation

In [2]:
import os
import torch
import torchvision
import torchvision.transforms as transforms

In [3]:
from google.colab import drive
from zipfile import ZipFile



drive.mount('/content/drive')
dataset_path = '/content/drive/MyDrive/datasets/tiny-imagenet-200.zip'
with ZipFile(dataset_path, "r") as zip_file:
    zip_file.extractall('/dataset')
print('Extraction complete!')

Mounted at /content/drive
Extraction complete!


We need to adjust the format of the val split of the dataset to be used with ImageFolder.

In [4]:
import os
import shutil

with open('/dataset/tiny-imagenet-200/val/val_annotations.txt') as f:
    for line in f:
        fn, cls, *_ = line.split('\t')
        os.makedirs(f'/dataset/tiny-imagenet-200/val/{cls}', exist_ok=True)

        shutil.copyfile(f'/dataset/tiny-imagenet-200/val/images/{fn}', f'/dataset/tiny-imagenet-200/val/{cls}/{fn}')

shutil.rmtree('/dataset/tiny-imagenet-200/val/images')

In [5]:
from torchvision.datasets import ImageFolder
import torchvision.transforms as T

transform = T.Compose([
    T.Resize((224, 224)),  # Resize to fit the input dimensions of the network
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# root/{classX}/x001.jpg

tiny_imagenet_dataset_train = ImageFolder(root='/dataset/tiny-imagenet-200/train', transform=transform)
tiny_imagenet_dataset_val = ImageFolder(root='/dataset/tiny-imagenet-200/val', transform=transform)

In [6]:
print(f"Length of train dataset: {len(tiny_imagenet_dataset_train)}")
print(f"Length of val dataset: {len(tiny_imagenet_dataset_val)}")

# The following code also checks the number of samples per class
#from collections import Counter

#class_counts = Counter([target for _, target in tiny_imagenet_dataset_val])
#for class_label, count in class_counts.items():
#    print(f"Class {class_label}: {count} entries")


Length of train dataset: 100000
Length of val dataset: 10000


In [7]:
train_loader = torch.utils.data.DataLoader(tiny_imagenet_dataset_train, batch_size=32, shuffle=True, num_workers=8)
val_loader = torch.utils.data.DataLoader(tiny_imagenet_dataset_val, batch_size=32, shuffle=False)

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:431: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


## Custom model definition

In [25]:
import torch
from torch import nn

# Define the custom neural network
class CustomNet(nn.Module):
    def __init__(self):
        super(CustomNet, self).__init__()
        # Define layers of the neural network
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(64, 64, kernel_size=3, padding=1)

        self.pool = nn.MaxPool2d(2, 2)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))

        self.fc1 = nn.Linear(64, 200)

    def forward(self, x):
        # Define forward pass

        # B x 3 x 224 x 224
        x = self.conv1(x).relu()
        x = self.pool(x)

        x = self.conv2(x).relu()

        x = self.avgpool(x)
        x = torch.flatten(x, start_dim=1)

        x = self.fc1(x)
        return x

In [9]:
def train(epoch, model, train_loader, criterion, optimizer):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for batch_idx, (X, y) in enumerate(train_loader):
        X, y = X.cuda(), y.cuda()

        pred = model(X)
        loss = criterion(pred, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        _, predicted = pred.max(1)
        total += y.size(0)
        correct += predicted.eq(y).sum().item()

    train_loss = running_loss / len(train_loader)
    train_accuracy = 100. * correct / total
    print(f'Train Epoch: {epoch} Loss: {train_loss:.6f} Acc: {train_accuracy:.2f}%')

In [10]:
# Validation loop
def validate(model, val_loader, criterion):
    model.eval()
    val_loss = 0

    correct, total = 0, 0

    with torch.no_grad():
        for batch_idx, (X, y) in enumerate(val_loader):
            X, y = X.cuda(), y.cuda()
            pred = model(X)
            loss = criterion(pred, y)
            val_loss += loss.item()
            _, predicted = pred.max(1)
            total += y.size(0)
            correct += predicted.eq(y).sum().item()

    val_loss = val_loss / len(val_loader)
    val_accuracy = 100. * correct / total

    print(f'Validation Loss: {val_loss:.6f} Acc: {val_accuracy:.2f}%')
    return val_accuracy

## Putting everything together

In [ ]:
model = CustomNet().cuda()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.001, momentum=0.9)

best_acc = 0

# Run the training process for {num_epochs} epochs
num_epochs = 10
for epoch in range(1, num_epochs + 1):
    train(epoch, model, train_loader, criterion, optimizer)

    # At the end of each training iteration, perform a validation step
    val_accuracy = validate(model, val_loader, criterion)

    # Best validation accuracy
    best_acc = max(best_acc, val_accuracy)


print(f'Best validation accuracy: {best_acc:.2f}%')


Train Epoch: 1 Loss: 5.255879 Acc: 1.01%
Validation Loss: 5.167996 Acc: 1.45%
Train Epoch: 2 Loss: 5.102615 Acc: 1.85%
Validation Loss: 5.046991 Acc: 2.36%
Train Epoch: 3 Loss: 5.012196 Acc: 2.72%
Validation Loss: 4.954324 Acc: 3.17%
Train Epoch: 4 Loss: 4.936446 Acc: 3.38%
Validation Loss: 4.905589 Acc: 3.45%
